In [20]:
import os, re, json, chromadb, uuid
from dotenv import load_dotenv

In [21]:
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in environment variables. Check your .env file.")

pine_api_key = os.getenv("PINNECONE_API_KEY")
if not pine_api_key:
    raise ValueError("PINNECONE_API_KEY not found in environment variables. Check your .env file.")

In [22]:
import os
from langchain_community.document_loaders import PyPDFLoader

def load_documents():
    folder_path = '../documents'
    all_docs = []
    num_docs = 0

    if not os.path.exists(folder_path):
        print(f"Error: Folder {folder_path} does not exist.")
        return []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path, filename)
            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            num_docs += 1
            all_docs.extend(doc)
            
    print(f"Successfully loaded {num_docs}.")
    print(f"Successfully loaded {len(all_docs)} total pages from PDFs.")
    return all_docs # Don't forget to return the list!

# Usage
documents = load_documents()


Successfully loaded 5.
Successfully loaded 36 total pages from PDFs.


In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs_in_chunks(documnets):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1000,
        chunk_overlap = 500
    )
    
    chunked_docs = text_splitter.split_documents(documnets)
    print(f"Total Chunks : {len(chunked_docs)}")
    return chunked_docs

chunked_docs = split_docs_in_chunks(documents)  
    

Total Chunks : 185


In [24]:
# for doc in chunked_docs:
#     print(doc.metadata.get("source", "unknown"))
#     print(doc.metadata.get("page_label", 0))

In [25]:
from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name=model_name
        print("Loading Model....", self.model_name)
        self.model=SentenceTransformer(self.model_name)
        print("Embeddings Dimensions : ", self.model.get_sentence_embedding_dimension())
        
    def generate_embedding(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("Embedding Shape : ", embeddings.shape)
        return embeddings     

In [26]:
embeddings_manager=EmbeddingManager()

Loading Model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings Dimensions :  384


In [27]:
import uuid
from pinecone import Pinecone, ServerlessSpec
import time

class PineconeManager:
    def __init__(self, api_key: str, index_name: str, dimension: int):
        self.pc = Pinecone(api_key=api_key) 
        self.index_name = index_name
        self.dimension = dimension
        self._initialize_index()
        self.index = self.pc.Index(index_name)
        
    def _initialize_index(self):
        existing_indexes = [idx.name for idx in self.pc.list_indexes()]
        if self.index_name not in existing_indexes:
            print(f"Creating index: {self.index_name}...")
            self.pc.create_index(
                name=self.index_name,
                dimension=self.dimension,
                metric="cosine",
                spec=ServerlessSpec(cloud="aws", region="us-east-1")
            )
            while not self.pc.describe_index(self.index_name).status['ready']:
                time.sleep(1)
        else:
            print(f"Connecting to existing index: {self.index_name}")
            
    def upsert_embeddings(self, chunks, embeddings_manager, namespace="pdf-documents"):
        vectors_to_upsert = [] 
        for doc in chunks:
            # Generate embedding
            vectors = embeddings_manager.generate_embedding(doc.page_content).tolist()
            vectors_id = str(uuid.uuid4())
            vector_metadata = {
                "text": doc.page_content,
                "source": doc.metadata.get("source", "unknown"),
                "page": doc.metadata.get("page_label", 0)
            }
            vectors_to_upsert.append((vectors_id, vectors, vector_metadata))
            
            # Batch upsert every 100 vectors
            if len(vectors_to_upsert) >= 100:
                self.index.upsert(vectors=vectors_to_upsert, namespace=namespace)
                vectors_to_upsert = []
                
        if len(vectors_to_upsert) > 0:
            self.index.upsert(vectors=vectors_to_upsert, namespace=namespace)
            
        print(f"Successfully uploaded {len(chunks)} chunks to Pinecone.")


In [28]:
# pc_manager=PineconeManager(api_key=pine_api_key, index_name="pdf-search-index", dimension=384)
# pc_manager.upsert_embeddings(chunked_docs, embeddings_manager)